In [ ]:
from pathlib import Path
import time
import json
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from skimage.metrics import peak_signal_noise_ratio, structural_similarity

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PROCESSED_DIR = Path("../data/processed")
PHYSICS_CKPT = Path("../outputs/checkpoints/best_physics_unet_hybrid.pth")
SRCNN_CKPT = Path("../outputs/checkpoints/best_srcnn_enhancement_hybrid.pth")

BATCH_SIZE = 1

print("Device:", device)
print("Physics checkpoint:", PHYSICS_CKPT.exists())
print("SRCNN checkpoint:", SRCNN_CKPT.exists())

In [ ]:
class LandsatPatchDataset(Dataset):
    def __init__(self, processed_dir, split):
        self.input_dir = Path(processed_dir) / split / "inputs"
        self.target_dir = Path(processed_dir) / split / "targets"

        self.input_files = sorted(self.input_dir.glob("*.npy"))
        self.target_files = sorted(self.target_dir.glob("*.npy"))

        assert len(self.input_files) == len(self.target_files)

    def __len__(self):
        return len(self.input_files)

    def __getitem__(self, idx):
        inp = np.load(self.input_files[idx]).astype(np.float32)
        tgt = np.load(self.target_files[idx]).astype(np.float32)

        inp = torch.from_numpy(inp).permute(2, 0, 1)
        tgt = torch.from_numpy(tgt).permute(2, 0, 1)

        return inp, tgt

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class PhysicsUNet(nn.Module):
    def __init__(self, in_channels=5, out_channels=3):
        super().__init__()

        self.down1 = DoubleConv(in_channels, 32)
        self.pool1 = nn.MaxPool2d(2)

        self.down2 = DoubleConv(32, 64)
        self.pool2 = nn.MaxPool2d(2)

        self.down3 = DoubleConv(64, 128)
        self.pool3 = nn.MaxPool2d(2)

        self.down4 = DoubleConv(128, 256)
        self.pool4 = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(256, 512)

        self.up4 = nn.ConvTranspose2d(512, 256, 2, 2)
        self.conv4 = DoubleConv(512, 256)

        self.up3 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.conv3 = DoubleConv(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.conv2 = DoubleConv(128, 64)

        self.up1 = nn.ConvTranspose2d(64, 32, 2, 2)
        self.conv1 = DoubleConv(64, 32)

        self.out = nn.Conv2d(32, out_channels, 1)
        self.activation = nn.Sigmoid()

    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(self.pool1(d1))
        d3 = self.down3(self.pool2(d2))
        d4 = self.down4(self.pool3(d3))

        bn = self.bottleneck(self.pool4(d4))

        u4 = self.up4(bn)
        u4 = self.conv4(torch.cat([u4, d4], dim=1))

        u3 = self.up3(u4)
        u3 = self.conv3(torch.cat([u3, d3], dim=1))

        u2 = self.up2(u3)
        u2 = self.conv2(torch.cat([u2, d2], dim=1))

        u1 = self.up1(u2)
        u1 = self.conv1(torch.cat([u1, d1], dim=1))

        return self.activation(self.out(u1))

In [ ]:
class SRCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=9, padding=4),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 32, kernel_size=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 3, kernel_size=5, padding=2),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
physics_model = PhysicsUNet(in_channels=5, out_channels=3).to(device)
srcnn_model = SRCNN().to(device)

physics_ckpt = torch.load(PHYSICS_CKPT, map_location=device)
srcnn_ckpt = torch.load(SRCNN_CKPT, map_location=device)

physics_model.load_state_dict(physics_ckpt["model_state_dict"])
srcnn_model.load_state_dict(srcnn_ckpt["model_state_dict"])

physics_model.eval()
srcnn_model.eval()

print("Loaded Physics Model:", physics_ckpt.get("model_name"))
print("Loaded SRCNN Model:", srcnn_ckpt.get("model_name"))

In [ ]:
test_dataset = LandsatPatchDataset(PROCESSED_DIR, "test")

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print("Test samples:", len(test_dataset))

In [ ]:
raw_psnr_scores = []
raw_ssim_scores = []

srcnn_psnr_scores = []
srcnn_ssim_scores = []

physics_times = []
srcnn_times = []
total_times = []

with torch.no_grad():
    for inp, tgt in tqdm(test_loader):
        inp = inp.to(device, non_blocking=True)
        tgt = tgt.to(device, non_blocking=True)

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        start_physics = time.time()
        raw_pred = physics_model(inp)

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        end_physics = time.time()

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        start_srcnn = time.time()
        enhanced_pred = srcnn_model(raw_pred)

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        end_srcnn = time.time()

        physics_times.append((end_physics - start_physics) * 1000)
        srcnn_times.append((end_srcnn - start_srcnn) * 1000)
        total_times.append((end_srcnn - start_physics) * 1000)

        raw_np = raw_pred.squeeze(0).cpu().permute(1, 2, 0).numpy()
        enhanced_np = enhanced_pred.squeeze(0).cpu().permute(1, 2, 0).numpy()
        tgt_np = tgt.squeeze(0).cpu().permute(1, 2, 0).numpy()

        raw_np = np.clip(raw_np, 0, 1)
        enhanced_np = np.clip(enhanced_np, 0, 1)
        tgt_np = np.clip(tgt_np, 0, 1)

        raw_psnr_scores.append(
            peak_signal_noise_ratio(tgt_np, raw_np, data_range=1.0)
        )

        raw_ssim_scores.append(
            structural_similarity(
                tgt_np,
                raw_np,
                channel_axis=-1,
                data_range=1.0
            )
        )

        srcnn_psnr_scores.append(
            peak_signal_noise_ratio(tgt_np, enhanced_np, data_range=1.0)
        )

        srcnn_ssim_scores.append(
            structural_similarity(
                tgt_np,
                enhanced_np,
                channel_axis=-1,
                data_range=1.0
            )
        )

print("Evaluation complete")

In [ ]:
raw_psnr = float(np.mean(raw_psnr_scores))
raw_ssim = float(np.mean(raw_ssim_scores))

srcnn_psnr = float(np.mean(srcnn_psnr_scores))
srcnn_ssim = float(np.mean(srcnn_ssim_scores))

avg_physics_ms = float(np.mean(physics_times))
avg_srcnn_ms = float(np.mean(srcnn_times))
avg_total_ms = float(np.mean(total_times))

fps = 1000.0 / avg_total_ms

print("=" * 70)
print("FINAL PIPELINE METRICS")
print("=" * 70)

print(f"Raw Physics U-Net PSNR:      {raw_psnr:.4f}")
print(f"Raw Physics U-Net SSIM:      {raw_ssim:.4f}")

print(f"SRCNN Enhanced PSNR:         {srcnn_psnr:.4f}")
print(f"SRCNN Enhanced SSIM:         {srcnn_ssim:.4f}")

print("-" * 70)
print(f"Physics U-Net Time:          {avg_physics_ms:.3f} ms/tile")
print(f"SRCNN Time:                  {avg_srcnn_ms:.3f} ms/tile")
print(f"Total Pipeline Time:         {avg_total_ms:.3f} ms/tile")
print(f"Approx FPS:                  {fps:.2f}")
print("=" * 70)

In [ ]:
@torch.no_grad()
def show_pipeline_result(idx=None):
    if idx is None:
        idx = np.random.randint(0, len(test_dataset))

    inp, tgt = test_dataset[idx]

    inp_batch = inp.unsqueeze(0).to(device)

    raw_pred = physics_model(inp_batch)
    enhanced_pred = srcnn_model(raw_pred)

    inp_np = inp.permute(1, 2, 0).numpy()
    tgt_np = tgt.permute(1, 2, 0).numpy()
    raw_np = raw_pred.squeeze(0).cpu().permute(1, 2, 0).numpy()
    enhanced_np = enhanced_pred.squeeze(0).cpu().permute(1, 2, 0).numpy()

    thermal = inp_np[:, :, 0]

    raw_error = np.abs(tgt_np - raw_np).mean(axis=-1)
    srcnn_error = np.abs(tgt_np - enhanced_np).mean(axis=-1)

    plt.figure(figsize=(22, 5))

    plt.subplot(1, 6, 1)
    plt.imshow(thermal, cmap="gray")
    plt.title("Input Thermal")
    plt.axis("off")

    plt.subplot(1, 6, 2)
    plt.imshow(np.clip(raw_np, 0, 1))
    plt.title("Physics U-Net Output")
    plt.axis("off")

    plt.subplot(1, 6, 3)
    plt.imshow(np.clip(enhanced_np, 0, 1))
    plt.title("SRCNN Enhanced")
    plt.axis("off")

    plt.subplot(1, 6, 4)
    plt.imshow(np.clip(tgt_np, 0, 1))
    plt.title("Ground Truth")
    plt.axis("off")

    plt.subplot(1, 6, 5)
    plt.imshow(raw_error, cmap="hot")
    plt.title("Raw Error")
    plt.axis("off")

    plt.subplot(1, 6, 6)
    plt.imshow(srcnn_error, cmap="hot")
    plt.title("SRCNN Error")
    plt.axis("off")

    plt.tight_layout()
    plt.show()


for _ in range(5):
    show_pipeline_result()

In [ ]:
metrics = {
    "raw_physics_unet_psnr": raw_psnr,
    "raw_physics_unet_ssim": raw_ssim,
    "srcnn_enhanced_psnr": srcnn_psnr,
    "srcnn_enhanced_ssim": srcnn_ssim,
    "physics_unet_ms_per_tile": avg_physics_ms,
    "srcnn_ms_per_tile": avg_srcnn_ms,
    "total_pipeline_ms_per_tile": avg_total_ms,
    "fps": fps,
    "physics_checkpoint": str(PHYSICS_CKPT),
    "srcnn_checkpoint": str(SRCNN_CKPT)
}

METRICS_DIR = Path("../outputs/metrics")
METRICS_DIR.mkdir(parents=True, exist_ok=True)

with open(METRICS_DIR / "final_pipeline_physics_srcnn_metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

print("Saved:", METRICS_DIR / "final_pipeline_physics_srcnn_metrics.json")

In [ ]:
## Final Pipeline will be without using SRCNN .